In [1]:
# pip install utm
# pip install neonutilities


In [61]:
import os
# This hides OMP Info and Warning messages
os.environ["KMP_WARNINGS"] = "0" 
os.environ["OMP_DISPLAY_ENV"] = "FALSE"

In [62]:
# load libraries
import geopandas as gpd

import requests
from pathlib import Path

import zipfile

import hvplot.pandas

import pandas as pd

import neonutilities as nu
import utm  

import xarray as xr
import hvplot.xarray  # Adds .hvplot() to xarray objects
import holoviews as hv
import rioxarray as rxr
import rioxarray.merge

import warnings


In [ ]:
# set base directory
base_dir = Path.home() / "Data" / "Earth Analytics" / "hyperspectral-proj"
base_dir.mkdir(parents=True, exist_ok=True)

In [4]:
url = "https://data.neonscience.org/api/v0/sites"
sites = requests.get(url).json()["data"]

df = pd.DataFrame(sites)

# show all Rocky Mountain-related entries
print(df[df["siteName"].str.contains("Rocky", na=False)][["siteCode", "siteName"]])

   siteCode              siteName
57     RMNP  Rocky Mountains NEON


In [ ]:
SITE_CODE = "RMNP"
PRODUCT_ID = "DP3.30015.001"
YEAR = "2022"

In [52]:
neon_dir = base_dir / f"neon_data_{SITE_CODE}"
neon_dir.mkdir(parents=True, exist_ok=True)
SAVE_PATH = neon_dir

In [24]:
def get_or_download_chm(lat, lon, year, site, save_path):
    # 1. Convert to UTM
    easting, northing, zone_num, _ = utm.from_latlon(lat, lon)
    tile_e = int(easting // 1000) * 1000
    tile_n = int(northing // 1000) * 1000
    
    print(f"Targeting: {tile_e} E, {tile_n} N (Zone {zone_num})")

    try:
        # 2. Attempt Download
        nu.by_tile_aop(
            dpid="DP3.30015.001",
            site=site,
            year=str(year),
            easting=tile_e,
            northing=tile_n,
            savepath=str(save_path),
            check_size=False,
            skip_if_exists=True
        )
        
        # 3. Locate the file
        search_pattern = f"**/NEON_*_{site}_DP3_{tile_e}_{tile_n}_CHM.tif"
        found_files = list(Path(save_path).glob(search_pattern))
        
        if found_files:
            return str(found_files[0])
        else:
            print(f"--- ERROR: No .tif file found in the download folder for {site} {year}. ---")
            return None

    except ValueError as e:
        if "Columns must be same length as key" in str(e):
            print(f"\n--- LOCATION ERROR ---")
            print(f"Coordinates ({lat}, {lon}) are outside the {site} flight box for {year}.")
            print(f"Try a coordinate closer to the site center, or check a different year.")
        else:
            print(f"An unexpected error occurred: {e}")
        return None

In [74]:
chm_file_path = get_or_download_chm(
    lat=40.275, 
    lon=-105.545, 
    year=2017, 
    site="RMNP", 
    save_path=neon_dir
)

Targeting: 453000 E, 4458000 N (Zone 13)


Provisional NEON data are not included. To download provisional data, use input parameter include_provisional=True.
Found 2 NEON data files totaling approximately 1.8 MB.
Files in savepath will be checked and skipped if they exist and match the latest version.
All files already exist locally and match the latest available data. Skipping download.


Tower Center: 40.2759, -105.5459

Safe Range: Keep Longitude between -105.50 and -105.58.
            Keep Latitude between 40.23 and 40.32

- Move North: Add +0.009 to Latitude
- Move South: Subtract -0.009 from Latitude
- Move East: Add +0.012 to Longitude
- Move West: Subtract -0.012 from Longitude

In [41]:
def plot_chm_on_map(tif_path, alpha_set = .7):
    # 1. Load using the new rioxarray engine
    # We use mask_and_scale=True to automatically handle those -9999 NoData values
    da = xr.open_dataset(tif_path, engine="rasterio", mask_and_scale=True).band_data.squeeze()
    
    # Optional: Keep it clean by removing the band dimension completely
    if 'band' in da.coords:
        da = da.drop_vars('band')

    # 2. Plot with hvplot
    plot = da.hvplot.image(
        x='x', y='y', 
        geo=True, 
        tiles='EsriImagery', 
        cmap='viridis', 
        alpha=alpha_set,
        title="NEON Canopy Height Model (RMNP)",
        clabel="Height (m)",
        width=700, height=500
    )
    
    return plot

In [75]:
# plots most recently download or called file
tile_plot = plot_chm_on_map(chm_file_path, alpha_set=.5)
tile_plot

:Overlay
   .WMTS.I  :WMTS   [Longitude,Latitude]
   .Image.I :Image   [x,y]   (band_data)

In [ ]:
# Download all CHM tiles for site and year
chm_all = nu.by_file_aop(
    dpid="DP3.30015.001",
    site="RMNP",
    year="2017",
    savepath=str(neon_dir),
    check_size=True,  # Recommended so you see the total size before starting
    skip_if_exists=True
)


In [79]:
# Suppress annoying coordinate system warnings if they clutter your output
# warnings.filterwarnings('ignore', message='.*specified chunks separate the stored chunks.*')

def merge_site_chm(data_dir, site_name):
    # 1. Find all CHM tiles
    tif_paths = [str(f) for f in Path(data_dir).rglob("*_CHM.tif")]
    
    if not tif_paths:
        print(f"No CHM tiles found in {data_dir}.")
        return None

    print(f"Merging {len(tif_paths)} tiles for {site_name}...")

    # 2. Load tiles with 'auto' chunking to avoid performance warnings
    tile_list = [
        xr.open_dataset(f, engine="rasterio", mask_and_scale=True, chunks={})
        .band_data.squeeze() 
        for f in tif_paths
    ]
    
    # 3. Use the correct merge function path
    merged_da = rioxarray.merge.merge_arrays(tile_list)

    return merged_da

In [80]:
# 2. Execute
full_chm = merge_site_chm(neon_dir, SITE_CODE)

Merging 233 tiles for RMNP...


In [82]:
def plot_full_chm(full_site_da, alpha_set=0.7):
    # 4. Interactive Map
    return full_site_da.hvplot.image(
        x='x', y='y', 
        geo=True, 
        tiles='EsriImagery', 
        cmap='viridis', 
        alpha=alpha_set,
        title=f"NEON Canopy Height Model: {SITE_CODE}",
        clabel="Height (m)",
        width=800, height=600,
        rasterize=True, 
        clim=(0, 50)
    )

In [85]:
site_map = plot_full_chm(full_chm)
site_map

BokehModel(combine_events=True, render_bundle={'docs_json': {'28aa03a0-77ea-4984-ad3e-b4bdb7296ecb': {'version…

In [95]:
hv.save(site_map, f'img/{SITE_CODE}_CHM_Map.html')
hv.save(tile_plot, f'img/{SITE_CODE}_CHM_Tile_Map.html')